# Heatwave-mean XGBoost scenario model (2020–2026)

This is a reproducible report front-end for the production helpers. The target is the **unweighted arithmetic mean of six clear daytime Landsat land-surface-temperature acquisitions at the same aligned 30 m location**. It is a six-acquisition composite—not a climatological normal, air temperature, a future forecast, or a temperature observed on one date. A location is excluded when any one of the six acquisitions is missing, non-finite, or not clear.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from greenwave_local_layers.image_regression import make_sector_folds
from greenwave_local_layers.image_regression_heatwave_mean import (
    HEATWAVE_OBSERVATION_IDS, prepare_heatwave_mean_catalog, validate_aligned_heatwave_grids, heatwave_raster_paths,
)
from greenwave_local_layers.image_regression_heatwave_mean_xgboost import ARTIFACTS, train_heatwave_mean_model
from greenwave_local_layers.image_regression_xgboost_pipeline import EMBARGO_METERS, OUTER_FOLDS, SEED
from greenwave_local_layers.sources import file_hash
plt.style.use('seaborn-v0_8-whitegrid')

## Source and grid validation

In [ ]:
grid = validate_aligned_heatwave_grids(heatwave_raster_paths())
catalog = prepare_heatwave_mean_catalog()
display(pd.DataFrame({
    'observation_id': HEATWAVE_OBSERVATION_IDS,
    'sha256': [catalog.manifest['sources']['landsat'][item]['sha256'] for item in HEATWAVE_OBSERVATION_IDS],
    'mean_c': [catalog.manifest['target']['perDateSummaries'][item]['meanC'] for item in HEATWAVE_OBSERVATION_IDS],
}))
grid

## Eligibility funnel and target

In [ ]:
eligibility = catalog.manifest['eligibility']
display(pd.DataFrame([
    {'stage': '2026 predictor-supported candidate centres', 'count': eligibility['candidateCount']},
    {'stage': 'Excluded: missing or unclear on ≥1 date', 'count': eligibility['excludedMissingAnyAcquisitionCount']},
    {'stage': 'Strict six-date complete cases', 'count': eligibility['eligibleCount']},
]))
assert len(catalog.samples) == 177_672 and catalog.samples.sector_id.nunique() == 154
catalog.manifest['target']['meanSummary']

In [ ]:
date_columns = [f"lst_{item.removeprefix('landsat-').replace('-', '_')}_c" for item in HEATWAVE_OBSERVATION_IDS]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
catalog.samples[date_columns].rename(columns=dict(zip(date_columns, HEATWAVE_OBSERVATION_IDS))).boxplot(ax=axes[0], rot=45)
axes[0].set_ylabel('Land-surface temperature (°C)'); axes[0].set_title('Per-acquisition target support')
axes[1].hist(catalog.samples.lst_c, bins=60, color='#176b87', alpha=.85)
axes[1].set_xlabel('Six-acquisition arithmetic mean LST (°C)'); axes[1].set_ylabel('Locations')
axes[1].set_title('Mean-target distribution')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
points = ax.scatter(catalog.samples.x_lambert, catalog.samples.y_lambert, c=catalog.samples.lst_c, s=.4, cmap='magma')
ax.set_aspect('equal'); ax.set_axis_off(); ax.set_title('Strict six-acquisition mean target at aligned 30 m locations')
fig.colorbar(points, ax=ax, label='Mean LST (°C)'); plt.tight_layout()

## Predictors and spatial folds

Predictors are the same scenario baseline used by the production 2026 model: Soil sealing 2024, Green Map 2021, and Urban Atlas 2021 fractions within 100 m. Their static dates are temporally mismatched with temperatures acquired from 2020 through 2026.

In [ ]:
from greenwave_local_layers.image_regression_heatwave_mean_xgboost import prepare_heatwave_mean_feature_cache
feature_path = prepare_heatwave_mean_feature_cache()
with np.load(feature_path, allow_pickle=False) as feature_cache:
    predictor_values = feature_cache['features'].copy()
    predictor_names = feature_cache['feature_names'].tolist()
predictor_summary = pd.DataFrame(predictor_values, columns=predictor_names).describe(percentiles=[.05, .5, .95]).T
display(predictor_summary)
fig, ax = plt.subplots(figsize=(13, 5))
ax.boxplot([predictor_values[:, index] for index in range(predictor_values.shape[1])], tick_labels=predictor_names, showfliers=False)
ax.tick_params(axis='x', rotation=70); ax.set_ylabel('Land-cover fraction'); ax.set_title('100 m radial predictor distributions')
plt.tight_layout()

In [ ]:
display(pd.DataFrame(catalog.manifest['predictorContract'].items(), columns=['contract', 'value']))
folds = make_sector_folds(catalog.samples, n_splits=OUTER_FOLDS, buffer_m=EMBARGO_METERS, seed=SEED)
display(pd.DataFrame([{
    'fold': fold.fold + 1, 'train': len(fold.train_indices), 'test': len(fold.test_indices),
    'embargo_excluded': len(fold.excluded_buffer_indices), 'test_sectors': len(fold.test_sector_ids),
} for fold in folds]))

## Independent model selection and final fit

The call below uses production code for seed-42 nested sector-held-out validation, the 200 m embargo, bounded recipe search, leakage-aware feature elimination, held-out predictions, the smoothing benchmark, and final artifact persistence. It does not reuse the 2026 model's selected parameters.

In [ ]:
if not ARTIFACTS.report.exists():
    report = train_heatwave_mean_model()
else:
    report = json.loads(ARTIFACTS.report.read_text(encoding='utf-8'))
display(pd.DataFrame(report['outerFolds']).loc[:, ['fold', 'rawMetrics', 'smoothedMetrics', 'selectedSigmaMeters']])
display(pd.Series(report['pooledOuterMetrics'], name='selected held-out pipeline'))

In [ ]:
final = report['final']
display(pd.Series({
    'retained_features': final['retainedFeatures'],
    'rejected_features': final['rejectedFeatures'],
    'boost_rounds': final['boostRounds'],
    'spatial_cv_rmse_c': final['spatialCvRmseC'],
    'smoothing_candidate_sigma_m': final['smoothingCandidateSigmaMeters'],
    'smoothing_promoted': final['smoothingPromoted'],
    'production_sigma_m': final['smoothingSigmaMeters'],
}))
display(pd.DataFrame(final['featureSelection']))

## Held-out predictions, residuals, and smoothing decision

In [ ]:
with np.load(ARTIFACTS.outer_predictions, allow_pickle=False) as predictions:
    observed = predictions['observed_c'].copy(); predicted = predictions['predicted_c'].copy(); residual = predictions['residual_c'].copy()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hexbin(observed, predicted, gridsize=55, mincnt=1, cmap='viridis')
limits = [min(observed.min(), predicted.min()), max(observed.max(), predicted.max())]
axes[0].plot(limits, limits, '--', color='#8f1d35'); axes[0].set(xlabel='Observed mean LST (°C)', ylabel='Held-out prediction (°C)')
axes[1].hist(residual, bins=60, color='#7b3fb4', alpha=.8); axes[1].axvline(0, color='black', lw=1)
axes[1].set(xlabel='Prediction − observation (°C)', ylabel='Locations')
plt.tight_layout()
display(pd.Series(report['smoothingBenchmark']['bootstrap']))

## Final artifact hashes and interpretation

These artifacts support an associative counterfactual scenario only. Each ΔLST is the model's modified land-cover prediction minus that same model's verified baseline prediction. Complete-case exclusion, predictor-year mismatch, residual spatial dependence, extrapolation outside training ranges, and the non-causal design remain limitations.

In [ ]:
artifacts = [ARTIFACTS.model, ARTIFACTS.report, ARTIFACTS.outer_predictions, ARTIFACTS.inference_grid, catalog.cache_dir / 'manifest.json']
display(pd.DataFrame([{'artifact': str(path), 'sha256': file_hash(path)} for path in artifacts]))
assert file_hash(ARTIFACTS.model) == report['final']['modelSha256']
assert file_hash(ARTIFACTS.inference_grid) == report['inferenceGrid']['sha256']